In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user's primary goal is to gather information about Lunapolis, including its capital, weather, cheese miners' population, and potential labor strikes.\n\n## SUMMARY\n\nThe capital of the moon is Lunapolis. The current weather in Lunapolis is clear, with temperatures ranging from a high of 120°C to a low of -100°C. There are 100,000 cheese miners living in Lunapolis, and it is predicted that the cheese miners' union will strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='46774eb5-1179-4e15-b1a5-ce5d0a453220'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='27791ddf-75f6-4805-9eda-a38a12932d8b'),
              AIMessage(content

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user's primary goal is to gather information about Lunapolis, including its capital, weather, cheese miners' population, and potential labor strikes.

## SUMMARY

The capital of the moon is Lunapolis. The current weather in Lunapolis is clear, with temperatures ranging from a high of 120°C to a low of -100°C. There are 100,000 cheese miners living in Lunapolis, and it is predicted that the cheese miners' union will strike due to dissatisfaction with the new president.

## ARTIFACTS

None

## NEXT STEPS

None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [6]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is seeking troubleshooting assistance for a device that won't turn on.\n\n## SUMMARY\nThe user reported that their device is plugged in and turned on but is not functioning. A diagnostic ping was initiated, confirming the device's temperature at 42°C and voltage at 2.9V. The next troubleshooting step involves checking for any lights or indicators on the device.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nAsk the user if the device is showing any lights or indicators.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='5dcb55ee-40d5-48b8-90a6-f36c4f69d1d1'),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='1e53556c-ac16-4f95-bd36-4ee3d020760b'),
              AIMessage(content='The diagnostic ping shows a temperature of 42°C.\n\nDo you see any lights or indicators on the device? 

In [7]:
print(response["messages"][-1].content)

The diagnostic ping shows a temperature of 42°C.

Do you see any lights or indicators on the device? If so, what color and is the light steady or blinking? If there are no lights, we can proceed to check the power connection and other steps.
